In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

# # # # # # # # # # # 
# tag = '2022041804_04'
# tag = '2022041807_a03' 
tag = '2022041809_a04'

## 1) Get latent codes.

In [ ]:
from eval_functions import *

# param_list = [('train', False, False), # train set (2000*6)
#               ('01', False, False),    # test set (2000*6)
#               ('01', True, False),     # ood set (2000*6)
#               ('01', False, True)]     # extended set (12000)


    
which_test = '01'
test_ood = False
test_extended = False

which_train = '01'
samp_size = 2000
n_epochs = 30

eval_bs = 6
use_cuda = True
empty_cuda = True
cuda_ids = [0,1,2,3]

df, latents = get_df_and_latents(tag, which_train, which_test, test_ood, test_extended,
                                 samp_size, eval_bs, n_epochs, use_cuda, empty_cuda, cuda_ids)

In [ ]:
mdir = '/home/kat/Repos/SALSA/data/model_ready/'

anc_path = f'{mdir}{which_test}/test/anchor_smiles.csv'
aug_path = f'{mdir}{which_test}/test/augmented_smiles.csv'   
# ds = ContraSeqDataset(anc_path, aug_path)
ds_arr = get_dataset_array(anc_path, aug_path)
anc_map = get_anc_map(ds_arr)

## 2) Get all pairwise dists.

In [ ]:
from tqdm.notebook import tqdm
from scipy.spatial import distance
from itertools import combinations as combo

all_dists = []

for idc_pair in tqdm(combo(range(len(latents)),2), total=len(latents)**2//2):
    idx1, idx2 = idc_pair
    if idx1==idx2:
        continue
    latent1 = latents[idx1]    
    latent2 = latents[idx2]    
    d = distance.euclidean(latent1, latent2)
    all_dists.append(d)

In [ ]:
len(all_dists)

## 3) Get anc-aug dists.

In [ ]:
from tqdm.notebook import tqdm
from scipy.spatial import distance

anc_aug_dists = []

for anc,augs in tqdm(anc_map.items(), total=len(anc_map)):
    augs = list(augs)
    augs.pop(0)
    
    latent_anc = latents[anc]    
    for aug in augs:
        latent_aug = latents[aug]
        d = distance.euclidean(latent_anc, latent_aug)
        anc_aug_dists.append(d)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='ticks',font_scale=1.25,palette='muted')

plt.figure(figsize=(15,3))
sns.histplot(anc_aug_dists, bins=200, kde=True)
plt.show()